[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sear-labs/energy-system-modeling/blob/main/notebooks/p5_storage_supply/19_battery_day.ipynb)

# What a battery is worth: one day, three questions

Sections 19.4 and 19.5 of the book, on a day you may already have built. [`teaching-code` notebook 12](https://github.com/sear-labs/teaching-code/blob/main/notebooks/12_energy_systems_pypsa/dispatch_to_pypsa.ipynb), Part E, builds it step by step: the demand, 140 MW of solar, a gas unit and a peaker, and a 40 MW / 160 MWh battery, with every storage equation PyPSA writes laid out. **Start there if you have not.** Here the same day is built in one cell, and the notebook asks what that build alone does not: what the battery is worth, what one line of its definition is doing, and whether a bigger one is worth more.

In Colab, *File › Save a copy in Drive* first so your edits are kept.

**Before you run anything:** if the battery held twice as much energy - 8 hours instead of 4 - would it save the system twice as much? Write down a yes or no and one reason.


## Setup

One cell, the same in every notebook here: it installs what Colab does not have, fetches this repository so `data/` and `src/` are present, and moves into this notebook's own folder so the relative paths below resolve. The notebook's own imports follow in the same cell.


In [1]:
# --- setup: generated by tools/sync_setup_cells.py -- do not edit here, edit that
# The same cell in every notebook in this series. It installs what Colab does
# not have, fetches the repository so that data/ and src/ are present, and moves
# into this notebook's own folder so the relative paths below resolve.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/sear-labs/energy-system-modeling"
REPO_NAME = REPO_URL.rstrip("/").split("/")[-1]
NOTEBOOK_DIR = "notebooks/p5_storage_supply"

# Pinned, per Part 1 rule 3: an unpinned install will one day pull a major
# version with a changed API and either break or silently alter the answer.
PINS = ["highspy>=1.11,<2", "pypsa>=1.3,<2"]

if "google.colab" in sys.modules:
    if PINS:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    # An ABSOLUTE base, so re-running this cell is safe. Colab's "Run all" is
    # commonly run twice, and a relative check would look for the clone inside
    # the folder it had already moved into -- cloning a second copy nested one
    # level down, then working from the wrong one.
    BASE = Path("/content") if Path("/content").is_dir() else Path.home()
    REPO_DIR = BASE / REPO_NAME
    if not REPO_DIR.exists():
        cloned = subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)],
            capture_output=True, text=True)
        if cloned.returncode != 0:
            raise SystemExit(
                "Could not clone " + REPO_URL + "\n"
                + (cloned.stderr or "").strip() + "\n\n"
                "If that says 'not found', the repository is still private.\n"
                "A raw file URL fails the same way, so there is no way around\n"
                "it: it has to be public before a student can run this.")
    os.chdir(REPO_DIR / NOTEBOOK_DIR)
elif Path.cwd().name != Path(NOTEBOOK_DIR).name:
    raise SystemExit(
        "Run this notebook from its own folder (" + NOTEBOOK_DIR + "),\n"
        "so that ../../src and ../../data resolve.")

sys.path.insert(0, str(Path.cwd().parents[1] / "src"))
print("working directory:", Path.cwd().name)

# --- end generated setup; the notebook's own imports follow ---

import numpy as np
import pandas as pd
import pypsa

working directory: p5_storage_supply


## The day

Notebook 12's demand and solar shapes, as two formulas.


In [2]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)  # PyPSA's notices about its next version

hours = np.arange(24)
demand_t = (50 + 60 * np.exp(-((hours - 19) ** 2) / 12.0)     # evening peak
              + 18 * np.exp(-((hours - 8) ** 2) / 6.0))      # morning bump
solar_pu = np.clip(np.sin(np.pi * (hours - 6) / 13), 0, 1)
print(f'peak demand {demand_t.max():.0f} MW at hour {demand_t.argmax()}; '
      f'140 MW of solar peaks at {140 * solar_pu.max():.0f} MW')


peak demand 110 MW at hour 19; 140 MW of solar peaks at 139 MW


## Part A - The same day, without a battery and with one


In [3]:
n = pypsa.Network()
n.set_snapshots(hours)
n.add('Bus', 'node')
n.add('Load', 'demand', bus='node', p_set=demand_t)
n.add('Generator', 'solar',  bus='node', p_nom=140, p_max_pu=solar_pu, marginal_cost=0)
n.add('Generator', 'gas',    bus='node', p_nom=66,  marginal_cost=22.14)
n.add('Generator', 'peaker', bus='node', p_nom=100, marginal_cost=80.00)

n.optimize(solver_name='highs')
cost_without = n.objective
print(f'day cost without a battery: ${cost_without:,.0f}')


Index(['node'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.04s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 72 primals, 168 duals
Objective: 2.85e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper were not assigned to the network.


day cost without a battery: $28,486


Now the battery, exactly as notebook 12 defines it.


In [4]:
n.add('StorageUnit', 'battery', bus='node', p_nom=40, max_hours=4,
      efficiency_store=0.927, efficiency_dispatch=0.927,
      standing_loss=0.01, cyclic_state_of_charge=True, marginal_cost=0.01)

n.optimize(solver_name='highs')
cost_with = n.objective
print(f'day cost with it: ${cost_with:,.0f}   saves ${cost_without - cost_with:,.0f}')


Index(['node'], dtype='object', name='name')


Index(['0'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.07s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 144 primals, 336 duals
Objective: 1.81e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


day cost with it: $18,132   saves $10,353


When it works: positive is discharging, negative is charging; the state of charge is in MWh.


In [5]:
pd.DataFrame({'battery MW': n.storage_units_t.p['battery'].round(1),
              'soc MWh': n.storage_units_t.state_of_charge['battery'].round(1),
              'peaker MW': n.generators_t.p['peaker'].round(1)}).T


snapshot,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
battery MW,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-40.0,-40.0,-14.5,18.8,5.7,40.0,39.2,27.0,12.3,0.0
soc MWh,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,...,112.1,148.0,160.0,138.1,130.6,86.1,43.0,13.4,-0.0,-0.0
peaker MW,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,-0.0,...,-0.0,-0.0,-0.0,-0.0,-0.0,4.0,-0.0,-0.0,-0.0,-0.0


> It fills while the sun is up and empties after dark. **Which hours it charges in is not a result:** solar is spilling through the middle of the day, so several schedules cost exactly the same and the solver hands back one of them - notebook 12 shows why. Read the table for the pattern. Then: which generator's megawatt-hours is the battery replacing - and is that the same thing as the hours with the highest price?


## Part B - Take away one line

`cyclic_state_of_charge=True` makes the battery finish the day where it started. Switch it off, and let the battery start the day full.


In [6]:
n.storage_units.loc['battery', 'cyclic_state_of_charge'] = False
n.storage_units.loc['battery', 'state_of_charge_initial'] = 160    # full, MWh

n.optimize(solver_name='highs')
cost_open = n.objective
print(f'day cost: ${cost_open:,.0f}   (with the cyclic rule: ${cost_with:,.0f})')
print('state of charge at 11 pm:',
      round(n.storage_units_t.state_of_charge['battery'].iloc[-1], 1), 'MWh')


Index(['node'], dtype='object', name='name')


Index(['0'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.09s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 144 primals, 336 duals
Objective: 1.49e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


day cost: $14,895   (with the cyclic rule: $18,132)
state of charge at 11 pm: -0.0 MWh


> The day got cheaper. Where did that energy come from, and who pays for it tomorrow? If you were selling this battery to someone, which of the two day costs would you show them?


## Part C - Longer batteries

Put the cyclic rule back, then try 2, 4 and 8 hours of storage at the same 40 MW. The loop repeats one identical step three times; nothing differs between its turns but the number of hours.


In [7]:
n.storage_units.loc['battery', 'cyclic_state_of_charge'] = True

rows = []
for h in (2, 4, 8):
    n.storage_units.loc['battery', 'max_hours'] = h
    n.optimize(solver_name='highs')
    rows.append({'hours': h, 'energy MWh': 40 * h,
                 'saves $/day': cost_without - n.objective})
pd.DataFrame(rows).round(0)


Index(['node'], dtype='object', name='name')


Index(['0'], dtype='object', name='name')


INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.08s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 144 primals, 336 duals
Objective: 2.27e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



...

INFO:linopy.model: Solve problem using Highs solver


INFO:linopy.io: Writing time: 0.07s


INFO:linopy.constants: Optimization successful: 
Status: ok
Termination condition: optimal
Solution: 144 primals, 336 duals
Objective: 1.63e+04
Solver: highs
Runtime: 0.00s
MIP gap: inf
Dual bound: 0.00e+00
Solver model: available
Solver message: Optimal



INFO:pypsa.optimization.optimize:The shadow-prices of the constraints Generator-fix-p-lower, Generator-fix-p-upper, StorageUnit-fix-p_dispatch-lower, StorageUnit-fix-p_dispatch-upper, StorageUnit-fix-p_store-lower, StorageUnit-fix-p_store-upper, StorageUnit-fix-state_of_charge-lower, StorageUnit-fix-state_of_charge-upper, StorageUnit-energy_balance were not assigned to the network.


,hours,energy MWh,saves $/day
0,2,80,5778.0
1,4,160,10353.0
2,8,320,12175.0


> Was your prediction right? Going from 4 to 8 hours added 160 MWh. What would that extra 160 MWh have to cost per year to be worth building? (The capital recovery factor of the book's section 9.3 turns a build cost into a yearly one.)


---
### Does the package agree?

`esm.storage` writes the storage equations out by hand as one linear program and solves it with `scipy` - the state of charge carried hour to hour, the standing loss, the cyclic rule or a starting charge. It takes this notebook's own numbers, read back from the network, and the day's shape from `data/raw/battery_day_profile.csv`, the same two formulas written to a table.

**Only the day's cost is compared**, never the schedule, for the reason in Part A. One detail was found rather than assumed: PyPSA applies the standing loss to each hour's charge from the hour before, but not to the starting charge in the first hour. Written the obvious way, Part B's two answers differ by about $32; written PyPSA's way, they agree.


In [8]:
from esm.storage import BatteryDay, day_cost, load_battery_profile
from esm.tolerance import AGREEMENT_RTOL, relative

g, b = n.generators, n.storage_units
profile_demand, profile_solar = load_battery_profile()
day = BatteryDay(demand=profile_demand, solar_pu=profile_solar,
                 solar_mw=g.at['solar', 'p_nom'],
                 gas_mw=g.at['gas', 'p_nom'], gas_cost=g.at['gas', 'marginal_cost'],
                 peaker_mw=g.at['peaker', 'p_nom'],
                 peaker_cost=g.at['peaker', 'marginal_cost'],
                 battery_mw=b.at['battery', 'p_nom'],
                 efficiency=b.at['battery', 'efficiency_store'],
                 standing_loss=b.at['battery', 'standing_loss'],
                 dispatch_cost=b.at['battery', 'marginal_cost'])

checks = [('no battery', cost_without, day_cost(day)[0]),
          ('4 h, cyclic', cost_with, day_cost(day, 4)[0]),
          ('4 h, starts full', cost_open, day_cost(day, 4, cyclic=False, soc_initial=160)[0])]
checks += [(f'saves, {r["hours"]} h', r['saves $/day'],
            cost_without - day_cost(day, r['hours'])[0]) for r in rows]
for label, hand, pkg in checks:
    print(f'{label:18s} {hand:12,.2f} {pkg:12,.2f}   {relative(hand, pkg):.1e}')

worst = max(relative(a, c) for _, a, c in checks)
assert worst < AGREEMENT_RTOL, f'notebook and package disagree by {worst:.2e}'
print(f'\nnotebook and package agree to {worst:.1e}')


no battery            28,485.74    28,485.74   0.0e+00
4 h, cyclic           18,132.39    18,132.39   0.0e+00
4 h, starts full      14,894.80    14,894.80   0.0e+00
saves, 2 h             5,777.83     5,777.83   6.3e-16
saves, 4 h            10,353.35    10,353.35   0.0e+00
saves, 8 h            12,175.10    12,175.10   0.0e+00

notebook and package agree to 6.3e-16


*Before class: the site in the book's Chapter 23 pays a demand charge on its single highest 15 minutes of the month. Would you size its battery by energy (MWh) or by power (MW) - and what would you look at to decide?*

### Sources
- System and battery: [`teaching-code` notebook 12](https://github.com/sear-labs/teaching-code/blob/main/notebooks/12_energy_systems_pypsa/dispatch_to_pypsa.ipynb), Part E (round trip 0.927² ≈ 86%, typical of lithium-ion; NREL ATB 2024).
